In [ ]:
from utils import *
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
import json

In [ ]:
def adaptive_ratio_loader():
    results_dir = Path("../results/adaptive_ratio")
    scalars = TBScalars(".cache/adaptive_ratio")

    res_df = []
    for test in tqdm([*results_dir.iterdir()]):
        params = test.name.split("-")
        test_r = {}
        test_r["env"] = params[0]
        types = {"seed": int}
        for (name, typ), value in zip(types.items(), params[1:]):
            test_r[name] = typ(value.removeprefix(f"{name}="))
        df = scalars.read(test)
        scores = df[df["tag"] == "val/mean_ep_ret"]["value"]
        test_r["score"] = scores.iloc[-1]
        res_df.append({"path": test, **test_r})
    res_df = pd.DataFrame.from_records(res_df)
    res_df

    return res_df, scalars


def adaptive_wm_ratio_loader():
    results_dir = Path("../results/adaptive_wm_ratio")
    scalars = TBScalars(".cache/adaptive_wm_ratio")

    res_df = []
    for test in tqdm([*results_dir.iterdir()]):
        params = test.name.split("-")
        test_r = {}
        test_r["env"] = params[0]
        types = {"seed": int}
        for (name, typ), value in zip(types.items(), params[1:]):
            test_r[name] = typ(value.removeprefix(f"{name}="))
        df = scalars.read(test)
        scores = df[df["tag"] == "val/mean_ep_ret"]["value"]
        test_r["score"] = scores.iloc[-1]
        res_df.append({"path": test, **test_r})
    res_df = pd.DataFrame.from_records(res_df)
    res_df

    return res_df, scalars


def adaptive_wm_ratio_v2_loader():
    res_df = []

    results_dir = Path("../results/adaptive_wm_ratio_v2")
    scalars = TBScalars(".cache/adaptive_wm_ratio_v2")
    for test in tqdm([*results_dir.iterdir()]):
        params = test.name.split("-")
        test_r = {"rl_ratio": 2}
        test_r["env"] = params[0]
        types = {"seed": int}
        for (name, typ), value in zip(types.items(), params[1:]):
            test_r[name] = typ(value.removeprefix(f"{name}="))
        df = scalars.read(test)
        scores = df[df["tag"] == "val/mean_ep_ret"]["value"]
        test_r["score"] = scores.iloc[-1]
        res_df.append({"path": test, **test_r})

    results_dir = Path("../results/adaptive_wm_ratio_v2_1")
    scalars = TBScalars(".cache/adaptive_wm_ratio_v2_1")
    for test in tqdm([*results_dir.iterdir()]):
        params = test.name.split("-")
        test_r = {}
        test_r["env"] = params[0]
        types = {"rl_ratio": int, "seed": int}
        for (name, typ), value in zip(types.items(), params[1:]):
            test_r[name] = typ(value.removeprefix(f"{name}="))
        df = scalars.read(test)
        scores = df[df["tag"] == "val/mean_ep_ret"]["value"]
        test_r["score"] = scores.iloc[-1]
        res_df.append({"path": test, **test_r})

    res_df = pd.DataFrame.from_records(res_df)

    return res_df, scalars


res_df, scalars = adaptive_ratio_loader()
res_wm_df, scalars_wm = adaptive_wm_ratio_loader()
res_wm2_df, scalars_wm2 = adaptive_wm_ratio_v2_loader()

In [ ]:
envs = res_wm2_df["env"].unique()

fig = make_subplots(
    rows=3,
    row_titles=[*envs],
    cols=1,
)

for row, env in enumerate(envs, 1):
    dfs = []
    df = res_df[res_df["env"] == env].copy()
    df["tag"] = "v0"
    dfs.append(df)
    df = res_wm_df[res_wm_df["env"] == env].copy()
    df["tag"] = "v1"
    dfs.append(df)
    df = res_wm2_df[res_wm2_df["env"] == env].copy()
    df["tag"] = [f"v2/{r['rl_ratio']}" for _, r in df.iterrows()]
    dfs.append(df)
    df = pd.concat(dfs, axis=0)
    for tag in sorted(df["tag"].unique()):
        fig.add_trace(
            go.Box(
                y=df[df["tag"] == tag]["score"],
                name=tag,
                boxmean="sd",
                boxpoints="all",
            ),
            row=row,
            col=1,
        )

fig.update_layout(width=1024, height=1500)
fig